# qwenlab runner (Kaggle)
The notebook only launches the scripts. All settings live in `configs/*.json`.
Set **Accelerator: GPU T4**, internet on, and add the qwenlab folder as a dataset input. Use *Save & Run All*.

In [ ]:
import os, shutil, subprocess
found=[p for p in subprocess.run(['find','/kaggle/input','-iname','pyproject.toml'],capture_output=True,text=True).stdout.split() if p]
SRC=found[0].rsplit('/',1)[0]
PKG='/kaggle/working/qwenlab_pkg'            # /kaggle/input is read-only: work from a copy
shutil.copytree(SRC, PKG, dirs_exist_ok=True)
print(PKG)
!pip install -q -e "$PKG" sentencepiece pyarrow
# optional Hub backup: add your token as a Kaggle secret named HF_TOKEN, then pass  train.hub_repo_id=user/repo  below
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no HF_TOKEN secret (fine unless you use hub_repo_id):', type(e).__name__)


In [ ]:
# 1. tests must pass before any GPU time is spent
!cd "$PKG" && python -m pytest tests -q

In [ ]:
# 2. train (English). Re-running this same cell resumes from the last checkpoint.
# Change the dataset by changing the second config file: configs/storytelling.json or configs/malayalam.json
!cd "$PKG" && python scripts/train.py --config configs/base.json configs/english.json --set train.output_dir=/kaggle/working/runs/english data.cache_dir=/kaggle/working/data_cache/english

In [ ]:
# 3. held-out loss + perplexity of the finished run
!cd "$PKG" && python scripts/evaluate.py --run-dir /kaggle/working/runs/english

In [ ]:
# 4. guarded generation (never longer than the training context window)
!cd "$PKG" && python scripts/generate.py --run-dir /kaggle/working/runs/english --prompt "The capital of France is"

In [ ]:
# 5. does repetition come from running past the context window, or from the model? (diagnostic)
!cd "$PKG" && python scripts/generate.py --run-dir /kaggle/working/runs/english --prompt "Once upon a time" --repetition-check